# Muraqam — full self-contained pipeline: train → ensemble → pseudo-label → retrain → submit

This trains **everything from scratch** in one Kaggle GPU run. No pre-existing checkpoints or OOF
pickles required. Pipeline:

1. **Stage A — base training.** Train all **4 members × 5 folds** with your solo-notebook config
   (LR 2e-5, OneCycle, focal γ=2, LLRD off, early-stop patience 4, AMP). Save fold checkpoints +
   full OOF per model.
2. **Ensemble OOF.** Fixed-weight blend `[1.3, 1.0, 2.0, 2.0]`, tune per-class thresholds on the
   pooled OOF (leakage-free), report honest CV macro-F1.
3. **Pseudo-label test.** Bag Stage-A folds on test, blend, threshold, then **combo-repair**: drop
   any predicted gap-combination never attested in train gold (subset-backoff — only removes hedged
   marks, never invents). Honorific `-ﷺ-` rule preserved.
4. **Stage B — augmented retrain.** One extra round: rebuild `train + pseudo`, **regenerate
   KFold(seed=2026)** over the combined set, retrain all 4 members 5-fold with the *same* config.
5. **Submit.** Bag Stage-B folds on test, blend, threshold, honorific rule → `submission.csv`.

Configs are **verbatim** from your solo notebook; metric/tokenizer/label/model/loss cells are copied
unchanged so there is zero train/scoring drift.

> **Runtime:** 4 models × 5 folds × 2 stages = 40 trainings. On a T4/P100 this is a multi-hour run
> — enable GPU and expect to let it run. `REUSE_SAVED=True` lets you resume: re-running skips any
> fold whose checkpoint already exists.

## 0. Config — solo-notebook settings, 4-model ensemble, both stages

In [ ]:
# =========================================================================
#  Self-contained config. Trains all members; no external checkpoint mounts.
# =========================================================================
!pip install -q transformers torch scikit-learn

import os, re, glob, math, json, random, pickle, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

TRAIN_CSV = "/kaggle/input/competitions/muraqqamchallenge/train.csv"
TEST_CSV  = "/kaggle/input/competitions/muraqqamchallenge/test.csv"
if not os.path.exists(TRAIN_CSV):
    TRAIN_CSV = "train.csv"; TEST_CSV = "test.csv"

MARKS = ['.', '،', '؟', '!', ':', '؛', '-']
M2I = {m: i for i, m in enumerate(MARKS)}
NUM_MARKS = len(MARKS)

# the four members (order aligns with MODEL_WEIGHTS)
MODELS = [
    "CAMeL-Lab/bert-base-arabic-camelbert-msa",
    "aubmindlab/bert-base-arabertv2",
    "makdadTaleb/arabic-punctuation-arabert",     # arabpunc
    "mahmoudmohammad/AraPUNCT",                    # AraELECTRA
]
MODEL_WEIGHTS = [1.3, 1.0, 2.0, 2.0]

# ---- training config: VERBATIM from your solo notebook ----
N_FOLDS     = 5
REUSE_SAVED = True          # resume: skip folds whose checkpoint already exists
GROUP_COL   = None          # doc-level KFold (set a column name for GroupKFold)
CHUNK_WORDS = 180
STRIDE      = 140
MAX_LEN     = 448
BATCH       = 8
LR             = 2e-5
WEIGHT_DECAY   = 0.01
FOCAL_GAMMA    = 2.0
WARMUP_FRAC    = 0.10
SCHEDULE       = "onecycle"
HEAD_DROPOUT   = 0.1
USE_LLRD       = False
LLRD_DECAY     = 0.95
USE_AMP             = True
EARLY_STOP_PATIENCE = 4
MAX_EPOCHS_CAP      = 30
THR_LO, THR_HI, THR_STEPS = 0.05, 0.90, 52
HONORIFICS = {"ﷺ"}; FORCE_HONORIFIC_RULE = True

# ---- stage dirs ----
DIR_A = "/kaggle/working/stageA_base"       # base folds + OOF
DIR_B = "/kaggle/working/stageB_augmented"  # retrained folds
PSEUDO_OUT = "/kaggle/working/pseudo"
for d in (DIR_A, DIR_B, PSEUDO_OUT): os.makedirs(d, exist_ok=True)

# ---- pseudo-label controls ----
DROP_UNSEEN_COMBOS = True
MIN_COMBO_SUPPORT  = 1      # >=2 treats train singletons as unseen (more aggressive repair)
CONF_GATE          = 0.0    # blank gaps whose max prob < gate; 0 = keep all
PSEUDO_ID_PREFIX   = "pseudo_"

print("device:", device, "| members:", len(MODELS), "| stages: A(base)+B(augmented)")
if device != "cuda":
    print("WARNING: no CUDA. Training 40 model-folds on CPU is not feasible — enable a GPU.")

## 1. Host metric (verbatim)

In [ ]:
# Host metric (verbatim) — for trustworthy LOCAL validation before you submit.
class ParticipantVisibleError(Exception):
    pass

"""
Kaggle metric for Arabic Punctuation Restoration.

Conventions:
    - `solution` is the full test CSV, including the hidden gold column.
    - `submission` is the competitor's CSV.
    - Both have a row_id column (already aligned & sorted by Kaggle).
    - `solution` has a column `raw` with the unpunctuated input and a column
      `gold` with the reference punctuated string.
    - `submission` has a column `prediction` with the competitor's
      punctuated string.

Scoring:
    Macro-F1 over the 7 Arabic sentence-punctuation classes
    ( . ، ؟ ! : ؛ - ), EXCLUDING the "no punctuation" class.

"""

import re
from typing import Optional

import pandas as pd
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer

# ---------------------------------------------------------------------------
# Classical Arabic punctuation-restoration set. Anything outside this is
# treated as word content (e.g. parentheses, quotes, numbers) and is NOT
# scored. Competitors must preserve those characters in their predictions.
# ---------------------------------------------------------------------------
VALID_SYMBOLS = set('.،؟!:؛-')


def _tokenize_gold(text: str):
    """
    Split a (gold or prediction) string into (leading_gap, [(word, trailing_gap), ...]).
    A "word" is a maximal run of non-whitespace, non-whitelist chars.
    A "gap" is any run of whitelist chars between words.
    """
    leading_gap_chars = []
    pairs = []
    current_word_chars = []
    in_word = False

    for ch in text:
        if ch.isspace():
            if in_word:
                pairs.append([''.join(current_word_chars), []])
                current_word_chars = []
                in_word = False
            continue

        if ch in VALID_SYMBOLS:
            if in_word:
                pairs.append([''.join(current_word_chars), [ch]])
                current_word_chars = []
                in_word = False
            else:
                if pairs:
                    pairs[-1][1].append(ch)
                else:
                    leading_gap_chars.append(ch)
            continue

        if not in_word:
            in_word = True
            current_word_chars = [ch]
        else:
            current_word_chars.append(ch)

    if in_word:
        pairs.append([''.join(current_word_chars), []])

    return ''.join(leading_gap_chars), [(w, ''.join(g)) for (w, g) in pairs]


def _extract_labels(raw_text: str, generated_text: str, role: str):
    """
    Align `generated_text` to `raw_text` word-by-word and return one list of
    symbols per word, representing the punctuation that appears in the gap
    after each word.

    Raises ValueError on any structural mismatch.
    """
    if raw_text is None or generated_text is None:
        raise ValueError(f"[{role}] text is empty or null")

    raw_words = str(raw_text).strip().split()
    if not raw_words:
        raise ValueError(f"[{role}] raw text contains no words")

    _, pairs = _tokenize_gold(str(generated_text))
    gen_words = [w for (w, _) in pairs]

    if len(gen_words) != len(raw_words):
        raise ValueError(
            f"[{role}] word-count mismatch: raw has {len(raw_words)} words, "
            f"{role} has {len(gen_words)}"
        )

    for i, (rw, gw) in enumerate(zip(raw_words, gen_words)):
        if rw != gw:
            raise ValueError(
                f"[{role}] word mismatch at position {i}: "
                f"raw='{rw}' vs {role}='{gw}'"
            )

    labels = []
    for _, gap in pairs:
        syms = [c for c in gap if c in VALID_SYMBOLS]
        labels.append(syms if syms else ['0'])

    return labels


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
    raw_column_name: str = 'text',
    gold_column_name: str = 'final_text',
    prediction_column_name: str = 'final_text',
) -> float:
    """
    Returns macro-F1 over the 7 Arabic punctuation
    classes, excluding the "no punctuation" class.
    """
    # --- 0. Drop row_id; Kaggle has already aligned the two frames -----------
    del solution[row_id_column_name]
    del submission[row_id_column_name]

    # --- 1. Column presence -------------------------------------------------
    for col in (raw_column_name, gold_column_name):
        if col not in solution.columns:
            # Organizer-side problem — hidden from competitor.
            raise RuntimeError(f"Solution is missing column '{col}'")
    if prediction_column_name not in submission.columns:
        raise ParticipantVisibleError(
            f"Submission is missing column '{prediction_column_name}'"
        )

    # --- 2. Length match ----------------------------------------------------
    if len(solution) != len(submission):
        raise ParticipantVisibleError(
            f"Submission has {len(submission)} rows, expected {len(solution)}"
        )

    # --- 3. Extract labels row-by-row --------------------------------------
    true_labels = []
    pred_labels = []

    raws = solution[raw_column_name].tolist()
    golds = solution[gold_column_name].tolist()
    preds = submission[prediction_column_name].tolist()

    for idx, (raw, gold, pred) in enumerate(zip(raws, golds, preds)):
        try:
            gold_seq = _extract_labels(raw, gold, role="gold")
        except ValueError as e:
            # Organizer-side: our own gold CSV is malformed for this row.
            raise RuntimeError(
                f"Gold extraction failed on row index {idx}: {e}"
            ) from e

        try:
            pred_seq = _extract_labels(raw, pred, role="prediction")
        except ValueError as e:
            # Competitor can fix this themselves.
            raise ParticipantVisibleError(
                f"Prediction at row index {idx} does not align with the raw "
                f"input. Your prediction must contain the same sequence of "
                f"non-punctuation words as the input, with only the allowed "
                f"punctuation symbols {sorted(VALID_SYMBOLS)} inserted "
                f"between them. Details: {e}"
            ) from e

        if len(gold_seq) != len(pred_seq):
            raise ParticipantVisibleError(
                f"Row {idx}: prediction has {len(pred_seq)} word positions "
                f"but the input has {len(gold_seq)}"
            )

        true_labels.extend(gold_seq)
        pred_labels.extend(pred_seq)

    # --- 4. Binarize using gold ∪ pred so hallucinated marks cost precision --
    all_classes = sorted(VALID_SYMBOLS) + ['0']
    mlb = MultiLabelBinarizer(classes=all_classes)
    y_true = mlb.fit_transform(true_labels)
    y_pred = mlb.transform(pred_labels)

    classes = list(mlb.classes_)
    zero_idx = classes.index('0')
    scored_cols = [i for i in range(len(classes)) if i != zero_idx]

    return float(f1_score(
        y_true[:, scored_cols],
        y_pred[:, scored_cols],
        average='macro',
        zero_division=0,
    ))

## 2. Tokenizer, labeling & honorific rule (verbatim)

In [ ]:
# =========================================================================
#  Word-gap tokenizer (IDENTICAL to the host metric) + labeling + rules
#  Sharing the metric's tokenizer guarantees zero train/scoring gap.
# =========================================================================
VALID_SYMBOLS = set('.،؟!:؛-')

def tokenize_gold(text):
    """-> (leading_gap, [(word, trailing_gap), ...]).  Matches host metric."""
    leading=[]; pairs=[]; cur=[]; in_word=False
    for ch in str(text):
        if ch.isspace():
            if in_word: pairs.append([''.join(cur), []]); cur=[]; in_word=False
            continue
        if ch in VALID_SYMBOLS:
            if in_word: pairs.append([''.join(cur), [ch]]); cur=[]; in_word=False
            else:
                if pairs: pairs[-1][1].append(ch)
                else: leading.append(ch)
            continue
        if not in_word: in_word=True; cur=[ch]
        else: cur.append(ch)
    if in_word: pairs.append([''.join(cur), []])
    return ''.join(leading), [(w, ''.join(g)) for w,g in pairs]

def gold_to_labels(final_text):
    """Return (words, Y) where Y is (n_words, NUM_MARKS) multi-hot of the gap AFTER each word."""
    _, pairs = tokenize_gold(final_text)
    words=[w for w,_ in pairs]
    Y=np.zeros((len(words), NUM_MARKS), dtype=np.float32)
    for i,(_,gap) in enumerate(pairs):
        for c in gap:
            if c in M2I: Y[i, M2I[c]] = 1.0
    return words, Y

def words_of_raw(raw):
    """Word units exactly as the metric derives them: whitespace split."""
    return str(raw).strip().split()

MARK_ORDER = MARKS  # canonical write order inside a gap (scoring is set-based anyway)

def reconstruct(words, pred_multi):
    """words + per-word multi-hot -> final_text string (word count preserved)."""
    toks=[]
    for w, row in zip(words, pred_multi):
        marks=''.join(m for m in MARK_ORDER if row[M2I[m]]>0)
        toks.append(w+marks)
    return ' '.join(toks)

def apply_honorific_rule(words, pred_multi):
    """Force -X- around each honorific: dash on its own gap AND the previous word's gap."""
    if not FORCE_HONORIFIC_RULE: return pred_multi
    P=pred_multi.copy()
    di=M2I['-']
    for i,w in enumerate(words):
        if w in HONORIFICS:
            P[i, di]=1.0
            if i>0: P[i-1, di]=1.0
    return P


## 3. Sliding-window dataset (verbatim)

In [ ]:
# =========================================================================
#  Sliding-window dataset. Punctuation is predicted at the LAST subword of
#  each word (the gap follows the word). Non-last subwords are masked.
# =========================================================================
def chunk_words(words, Y=None):
    """Yield (word_slice, Y_slice, start_index) windows over a long word list."""
    n=len(words)
    if n<=CHUNK_WORDS:
        yield words, (Y if Y is not None else None), 0; return
    s=0
    while s<n:
        e=min(s+CHUNK_WORDS, n)
        yield words[s:e], (Y[s:e] if Y is not None else None), s
        if e==n: break
        s+=STRIDE

class PunctDataset(Dataset):
    def __init__(self, rows, tokenizer, has_labels=True):
        self.samples=[]
        self.tok=tokenizer; self.has_labels=has_labels
        for r in rows:
            words=words_of_raw(r["text"])
            if has_labels:
                gw,Y=gold_to_labels(r["final_text"])
                # words already verified equal to gw
            else:
                Y=None
            for wslice,Yslice,start in chunk_words(words,Y):
                self.samples.append((wslice,Yslice))
    def __len__(self): return len(self.samples)
    def __getitem__(self,i):
        words,Y=self.samples[i]
        enc=self.tok(words, is_split_into_words=True, truncation=True,
                     max_length=MAX_LEN, return_tensors=None)
        word_ids=enc.word_ids()
        # mark the LAST subword of each word as the active prediction position
        last_pos={}
        for pos,wid in enumerate(word_ids):
            if wid is not None: last_pos[wid]=pos
        active=np.zeros(len(word_ids),dtype=bool)
        labels=np.zeros((len(word_ids),NUM_MARKS),dtype=np.float32)
        wid_at=np.full(len(word_ids),-1,dtype=np.int64)
        for wid,pos in last_pos.items():
            active[pos]=True; wid_at[pos]=wid
            if Y is not None and wid<len(Y): labels[pos]=Y[wid]
        return {"input_ids":enc["input_ids"],"attention_mask":enc["attention_mask"],
                "active":active,"labels":labels,"word_ids":wid_at}

def collate(batch, pad_id):
    maxlen=max(len(b["input_ids"]) for b in batch)
    B=len(batch)
    input_ids=np.full((B,maxlen),pad_id,dtype=np.int64)
    attn=np.zeros((B,maxlen),dtype=np.int64)
    active=np.zeros((B,maxlen),dtype=bool)
    labels=np.zeros((B,maxlen,NUM_MARKS),dtype=np.float32)
    wids=np.full((B,maxlen),-1,dtype=np.int64)
    for i,b in enumerate(batch):
        L=len(b["input_ids"])
        input_ids[i,:L]=b["input_ids"]; attn[i,:L]=b["attention_mask"]
        active[i,:L]=b["active"]; labels[i,:L]=b["labels"]; wids[i,:L]=b["word_ids"]
    return (torch.tensor(input_ids),torch.tensor(attn),torch.tensor(active),
            torch.tensor(labels),torch.tensor(wids))
print("dataset utilities ready")

## 4. Multi-label model + focal loss (verbatim)

In [ ]:
# =========================================================================
#  Multi-LABEL token classifier (independent sigmoid per mark) + focal loss.
#  Multi-label (NOT 8-way softmax) is essential: gaps like ؟! carry two marks,
#  and the metric scores each mark independently. Softmax would forfeit them.
# =========================================================================
class PunctModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone=AutoModel.from_pretrained(model_name)
        h=self.backbone.config.hidden_size
        self.drop=nn.Dropout(globals().get("HEAD_DROPOUT",0.1))
        self.head=nn.Linear(h,NUM_MARKS)
    def forward(self,input_ids,attention_mask):
        out=self.backbone(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state
        return self.head(self.drop(out))          # (B,T,NUM_MARKS) logits

def focal_bce(logits, targets, active, gamma=FOCAL_GAMMA, pos_weight=None):
    """Focal binary cross-entropy, averaged over ACTIVE positions only."""
    logits=logits[active]; targets=targets[active]          # (N,NUM_MARKS)
    if logits.numel()==0:
        return logits.sum()*0.0
    bce=nn.functional.binary_cross_entropy_with_logits(
        logits,targets,reduction='none',pos_weight=pos_weight)
    p=torch.sigmoid(logits)
    p_t=p*targets+(1-p)*(1-targets)
    focal=((1-p_t)**gamma)*bce
    return focal.mean()

# per-mark positive weight from class frequency (rarer -> upweighted)
def compute_pos_weight(rows):
    pos=np.zeros(NUM_MARKS); tot=0
    for r in rows:
        _,Y=gold_to_labels(r["final_text"]); pos+=Y.sum(0); tot+=len(Y)
    neg=tot-pos
    w=np.clip(neg/np.clip(pos,1,None),1.0,20.0)   # cap to avoid instability
    return torch.tensor(w,dtype=torch.float32)
print("model + focal loss ready")

## 5. Train / inference utilities — AMP, early stopping, LLRD, schedule (verbatim)

In [ ]:
# =========================================================================
#  Training with AMP (mixed precision) + EARLY STOPPING on validation macro-F1.
#  No more guessing epoch counts: we train up to MAX_EPOCHS_CAP and keep the
#  weights from the best validation epoch (patience = EARLY_STOP_PATIENCE).
#  Validation signal = per-word macro-F1 at threshold 0.5 (cheap proxy of the
#  host metric; the real host-metric check still runs after threshold tuning).
# =========================================================================
from functools import partial

def _val_macro_f1(model, tok, val_rows):
    """Cheap per-word macro-F1 at 0.5 over the 7 marks (early-stopping signal)."""
    model.eval(); pad_id=tok.pad_token_id if tok.pad_token_id is not None else 0
    tp=np.zeros(NUM_MARKS); fp=np.zeros(NUM_MARKS); fn=np.zeros(NUM_MARKS)
    with torch.no_grad():
        for r in val_rows:
            words=words_of_raw(r["text"]); n=len(words)
            _,Y=gold_to_labels(r["final_text"])
            acc=np.zeros((n,NUM_MARKS)); cnt=np.zeros((n,1))+1e-9
            for wslice,_,start in chunk_words(words,None):
                enc=tok(wslice,is_split_into_words=True,truncation=True,max_length=MAX_LEN,return_tensors="pt")
                wid=enc.word_ids()
                logits=model(enc["input_ids"].to(device),enc["attention_mask"].to(device))[0]
                probs=torch.sigmoid(logits).float().cpu().numpy()
                last={}
                for pos,w in enumerate(wid):
                    if w is not None: last[w]=pos
                for w,pos in last.items():
                    gi=start+w
                    if gi<n: acc[gi]+=probs[pos]; cnt[gi]+=1
            pred=((acc/cnt)>=0.5).astype(int)
            tp+=((pred==1)&(Y==1)).sum(0); fp+=((pred==1)&(Y==0)).sum(0); fn+=((pred==0)&(Y==1)).sum(0)
    prec=tp/np.clip(tp+fp,1,None); rec=tp/np.clip(tp+fn,1,None)
    f1=np.where((prec+rec)>0,2*prec*rec/np.clip(prec+rec,1e-9,None),0.0)
    return float(f1.mean())

# ---- optimizer with optional layer-wise LR decay (LLRD) ----
def build_optimizer(model, base_lr, weight_decay):
    use_llrd = globals().get("USE_LLRD", False)
    if not use_llrd:
        return torch.optim.AdamW(model.parameters(), lr=base_lr, weight_decay=weight_decay)
    decay = globals().get("LLRD_DECAY", 0.95)
    # locate transformer layers (BERT family: backbone.encoder.layer)
    try:
        layers = model.backbone.encoder.layer
        n = len(layers)
    except Exception:
        return torch.optim.AdamW(model.parameters(), lr=base_lr, weight_decay=weight_decay)
    groups=[]; assigned=set()
    # head + non-backbone params at full LR
    head_params=[p for nme,p in model.named_parameters() if not nme.startswith("backbone.encoder.layer") and not nme.startswith("backbone.embeddings")]
    groups.append({"params":head_params,"lr":base_lr,"weight_decay":weight_decay})
    # each encoder layer: deeper layers -> higher LR, shallow -> lower
    for i,layer in enumerate(layers):
        lr_i = base_lr * (decay ** (n-1-i))
        groups.append({"params":list(layer.parameters()),"lr":lr_i,"weight_decay":weight_decay})
    # embeddings: lowest LR
    try:
        emb=list(model.backbone.embeddings.parameters())
        groups.append({"params":emb,"lr":base_lr*(decay**n),"weight_decay":weight_decay})
    except Exception: pass
    return torch.optim.AdamW(groups)

def build_scheduler(opt, total_steps, warmup_frac):
    sched_type = globals().get("SCHEDULE","onecycle")
    warm=int(total_steps*warmup_frac)
    if sched_type=="onecycle":
        lrs=[g["lr"] for g in opt.param_groups]
        return torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lrs, total_steps=total_steps, pct_start=warmup_frac)
    if sched_type in ("cosine","linear"):
        from torch.optim.lr_scheduler import LambdaLR
        import math
        def lr_lambda(step):
            if step<warm: return step/max(1,warm)
            prog=(step-warm)/max(1,total_steps-warm)
            if sched_type=="cosine": return max(0.0, 0.5*(1+math.cos(math.pi*prog)))
            return max(0.0, 1-prog)  # linear
        return LambdaLR(opt, lr_lambda)
    return torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=total_steps, pct_start=warmup_frac)

def train_one(model_name, train_rows, val_rows):
    tok=AutoTokenizer.from_pretrained(model_name)
    pad_id=tok.pad_token_id if tok.pad_token_id is not None else 0
    tr=PunctDataset(train_rows,tok,has_labels=True)
    dl=DataLoader(tr,batch_size=BATCH,shuffle=True,collate_fn=partial(collate,pad_id=pad_id))
    model=PunctModel(model_name).to(device)     # AutoModel loads encoder; head discarded
    pw=compute_pos_weight(train_rows).to(device)
    wd=WEIGHT_DECAY if "WEIGHT_DECAY" in globals() else 0.01
    opt=build_optimizer(model, LR, wd)
    n_epochs=MAX_EPOCHS_CAP
    total=len(dl)*n_epochs
    warm=WARMUP_FRAC if "WARMUP_FRAC" in globals() else 0.1
    sched=build_scheduler(opt, total, warm)
    scaler=torch.cuda.amp.GradScaler(enabled=(USE_AMP and device=="cuda"))

    best_f1=-1.0; best_state=None; patience=0
    for ep in range(n_epochs):
        model.train(); run=0.0
        for input_ids,attn,active,labels,_ in dl:
            input_ids,attn=input_ids.to(device),attn.to(device)
            active,labels=active.to(device),labels.to(device)
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=(USE_AMP and device=="cuda")):
                logits=model(input_ids,attn)
                loss=focal_bce(logits,labels,active,pos_weight=pw)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(opt); scaler.update(); sched.step(); run+=loss.item()
        vf1=_val_macro_f1(model,tok,val_rows) if val_rows else -1.0
        tag=f"  [{model_name.split('/')[-1]}] epoch {ep+1}/{n_epochs} loss {run/len(dl):.4f}"
        if val_rows:
            tag+=f" | val macroF1 {vf1:.4f}"
            if vf1>best_f1+1e-4:
                best_f1=vf1; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; patience=0; tag+="  *"
            else:
                patience+=1
        print(tag)
        if val_rows and patience>=EARLY_STOP_PATIENCE:
            print(f"    early stop (no val gain in {EARLY_STOP_PATIENCE} epochs); best macroF1 {best_f1:.4f}")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, tok

@torch.no_grad()
def predict_probs(model, tok, rows):
    model.eval(); pad_id=tok.pad_token_id if tok.pad_token_id is not None else 0
    results=[]
    for r in rows:
        words=words_of_raw(r["text"]); n=len(words)
        acc=np.zeros((n,NUM_MARKS)); cnt=np.zeros((n,1))+1e-9
        for wslice,_,start in chunk_words(words,None):
            enc=tok(wslice,is_split_into_words=True,truncation=True,max_length=MAX_LEN,return_tensors="pt")
            wid=enc.word_ids()
            logits=model(enc["input_ids"].to(device),enc["attention_mask"].to(device))[0]
            probs=torch.sigmoid(logits).float().cpu().numpy()
            last={}
            for pos,w in enumerate(wid):
                if w is not None: last[w]=pos
            for w,pos in last.items():
                gi=start+w
                if gi<n: acc[gi]+=probs[pos]; cnt[gi]+=1
        results.append((words, acc/cnt))
    return results

def ensemble_probs(list_of_results):
    """Weighted average of per-word probs across models (MODEL_WEIGHTS by index).
    Falls back to equal weights if MODEL_WEIGHTS is undefined."""
    w=np.array(MODEL_WEIGHTS,dtype=float) if "MODEL_WEIGHTS" in globals() else np.ones(len(list_of_results))
    w=w/ w.sum()
    base=list_of_results[0]; out=[]
    for k in range(len(base)):
        words=base[k][0]
        P=np.zeros_like(base[k][1])
        for j,lr in enumerate(list_of_results):
            P=P+w[j]*lr[k][1]
        out.append((words,P))
    return out

# ---- fold-checkpoint persistence (so reruns never retrain) ----
SAVE_DIR = globals().get("SAVE_DIR", "/kaggle/working/muraqam_cv")
os.makedirs(SAVE_DIR, exist_ok=True)
def _slug(mn): return mn.split("/")[-1].replace("-","_")
def ckpt_path(mn,k): return os.path.join(SAVE_DIR, f"{_slug(mn)}_fold{k}.pt")

def load_fold_model(mn,k):
    """Rebuild PunctModel and load a saved fold checkpoint (eval mode)."""
    model = PunctModel(mn).to(device)
    sd = torch.load(ckpt_path(mn,k), map_location=device)
    model.load_state_dict(sd); model.eval()
    tok = AutoTokenizer.from_pretrained(mn)
    return model, tok

print("AMP + early-stopping train/infer ready")

## 6. Per-class threshold tuning (verbatim)

In [ ]:
# =========================================================================
#  Per-class threshold search to maximize MACRO-F1.
#  In multi-label the classes are independent, so tuning each mark's threshold
#  separately is optimal for macro-F1. Rare marks get lower thresholds (recall).
# =========================================================================
def per_class_f1(y_true, y_pred):
    tp=((y_pred==1)&(y_true==1)).sum(0)
    fp=((y_pred==1)&(y_true==0)).sum(0)
    fn=((y_pred==0)&(y_true==1)).sum(0)
    prec=tp/np.clip(tp+fp,1,None); rec=tp/np.clip(tp+fn,1,None)
    f1=np.where((prec+rec)>0, 2*prec*rec/np.clip(prec+rec,1e-9,None), 0.0)
    return f1

def tune_thresholds(val_results, val_rows):
    # stack word-level gold + probs across all val rows
    golds=[]; probs=[]
    for (words,P),r in zip(val_results,val_rows):
        _,Y=gold_to_labels(r["final_text"])
        golds.append(Y); probs.append(P)
    Yt=np.concatenate(golds,0); Pp=np.concatenate(probs,0)
    grid=np.linspace(globals().get("THR_LO",0.10),globals().get("THR_HI",0.90),globals().get("THR_STEPS",33))
    best=np.full(NUM_MARKS,0.5)
    for m in range(NUM_MARKS):
        bf,bt=-1,0.5
        for t in grid:
            pred=(Pp[:,m]>=t).astype(int)
            tp=((pred==1)&(Yt[:,m]==1)).sum()
            fp=((pred==1)&(Yt[:,m]==0)).sum()
            fn=((pred==0)&(Yt[:,m]==1)).sum()
            prec=tp/max(tp+fp,1); rec=tp/max(tp+fn,1)
            f1=2*prec*rec/(prec+rec) if (prec+rec)>0 else 0
            if f1>bf: bf,bt=f1,t
        best[m]=bt
    return best

def probs_to_multihot(words, P, thresholds):
    pred=(P>=thresholds[None,:]).astype(np.float32)
    pred=apply_honorific_rule(words,pred)   # force -X- honorifics
    return pred
print("threshold tuning ready")

## 7. Generic 5-fold trainer

One function trains any member set over any dataframe that already carries a `fold` column, saving
`{slug}_fold{k}.pt` and returning the OOF. Used unchanged by both Stage A and Stage B.

In [ ]:
# =========================================================================
#  Reusable CV driver: trains MODELS x N_FOLDS over a foldted dataframe.
#  Saves checkpoints into `save_dir`; returns oof dict {model: [ (words,P) ]}.
# =========================================================================
def _slug(mn): return mn.split("/")[-1].replace("-", "_")

def run_cv(frame, save_dir, tag=""):
    def ckpt(mn,k): return os.path.join(save_dir, f"{_slug(mn)}_fold{k}.pt")
    def load_fold(mn,k):
        m = PunctModel(mn).to(device)
        m.load_state_dict(torch.load(ckpt(mn,k), map_location=device)); m.eval()
        return m, AutoTokenizer.from_pretrained(mn)

    oof = {mn: [None]*len(frame) for mn in MODELS}
    for mn in MODELS:
        print("="*72); print(f"[{tag}] MODEL:", mn)
        for k in range(N_FOLDS):
            va_idx = np.where(frame["fold"].values == k)[0]
            tr_idx = np.where(frame["fold"].values != k)[0]
            va_rows = [frame.iloc[i] for i in va_idx]
            tr_rows = [frame.iloc[i] for i in tr_idx]
            if REUSE_SAVED and os.path.exists(ckpt(mn,k)):
                print(f"  fold {k}: load saved ({len(va_idx)} OOF docs)")
                m, tok = load_fold(mn,k)
            else:
                print(f"  fold {k}: train on {len(tr_idx)} | OOF on {len(va_idx)}")
                m, tok = train_one(mn, tr_rows, va_rows)
                torch.save({kk:v.detach().cpu() for kk,v in m.state_dict().items()}, ckpt(mn,k))
                print(f"    saved -> {ckpt(mn,k)}")
            res = predict_probs(m, tok, va_rows)
            for j,i in enumerate(va_idx): oof[mn][i] = res[j]
            del m; gc.collect()
            if device=="cuda": torch.cuda.empty_cache()
    with open(os.path.join(save_dir,"oof_probs.pkl"),"wb") as f:
        pickle.dump({"MODELS":MODELS,"oof":oof,"fold":frame["fold"].values,
                     "ids":frame["id"].values}, f)
    frame[["id","fold"]].to_csv(os.path.join(save_dir,"fold_map.csv"), index=False)
    print(f"[{tag}] saved OOF + fold map -> {save_dir}")
    return oof

# fixed-weight blend helpers (shared by both stages)
_w = np.array(MODEL_WEIGHTS,float); _w = _w/_w.sum()
def blend_docs(per):                     # per: (n_models, n_words, 7)
    return np.clip((per*_w[:,None,None]).sum(0), 0, 1)
def ensemble_oof(oof, frame):
    out=[]
    for i in range(len(frame)):
        words = oof[MODELS[0]][i][0]
        per = np.stack([oof[mn][i][1] for mn in MODELS], axis=0)
        out.append((words, blend_docs(per)))
    return out
def bag_test(save_dir, test_rows):
    def ckpt(mn,k): return os.path.join(save_dir, f"{_slug(mn)}_fold{k}.pt")
    def load_fold(mn,k):
        m=PunctModel(mn).to(device)
        m.load_state_dict(torch.load(ckpt(mn,k),map_location=device)); m.eval()
        return m, AutoTokenizer.from_pretrained(mn)
    per_model={}
    for mn in MODELS:
        fr=[]
        for k in range(N_FOLDS):
            m,tok=load_fold(mn,k); fr.append(predict_probs(m,tok,test_rows))
            del m; gc.collect()
            if device=="cuda": torch.cuda.empty_cache()
        base=fr[0]; avg=[]
        for t in range(len(base)):
            words=base[t][0]; P=np.mean([f[t][1] for f in fr],axis=0); avg.append((words,P))
        per_model[mn]=avg; print("  bagged",N_FOLDS,"folds for",mn.split("/")[-1])
    out=[]
    for t in range(len(test_rows)):
        words=per_model[MODELS[0]][t][0]
        per=np.stack([per_model[mn][t][1] for mn in MODELS],axis=0)
        out.append((words, blend_docs(per)))
    return out
print("CV driver + blend helpers ready")

## 8. STAGE A — load train, make fresh 5 folds, train base 4 models

In [ ]:
# =========================================================================
#  Fresh KFold(seed=2026) over train, then train all members 5-fold (Stage A).
# =========================================================================
from sklearn.model_selection import KFold, GroupKFold

df = pd.read_csv(TRAIN_CSV).dropna(subset=["text","final_text"]).reset_index(drop=True)
fold_of = np.full(len(df), -1, int)
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for k,(_,va) in enumerate(kf.split(np.arange(len(df)))): fold_of[va]=k
df["fold"] = fold_of
print(f"KFold(seed={SEED}) -> docs/fold:", df["fold"].value_counts().sort_index().to_dict())

# alignment sanity (metric relies on gold words == raw words)
_bad=sum(gold_to_labels(df.iloc[i]["final_text"])[0]!=words_of_raw(df.iloc[i]["text"]) for i in range(len(df)))
print("word-alignment mismatches (want 0):", _bad)

oofA = run_cv(df, DIR_A, tag="A/base")

## 9. Ensemble Stage-A OOF, tune thresholds, honest CV

In [ ]:
# =========================================================================
#  Blend OOF, tune per-class thresholds on pooled OOF, report host macro-F1.
#  These thresholds binarize TEST into pseudo-labels.
# =========================================================================
oof_rows = [df.iloc[i] for i in range(len(df))]
ensA = ensemble_oof(oofA, df)
thresholds = tune_thresholds(ensA, oof_rows)
np.save(os.path.join(PSEUDO_OUT,"thresholds.npy"), thresholds)
print("tuned thresholds:", {MARKS[i]:round(float(thresholds[i]),3) for i in range(NUM_MARKS)})

textsA=[reconstruct(wd, probs_to_multihot(wd,P,thresholds)) for (wd,P) in ensA]
solA=pd.DataFrame([{"id":i,"text":df.iloc[i]["text"],"final_text":df.iloc[i]["final_text"]} for i in range(len(df))])
subA=pd.DataFrame({"id":range(len(df)),"final_text":textsA})
macroA=score(solA.copy(), subA.copy(), "id")
print(f"\n>>> Stage-A OOF host macro-F1 (4-model ensemble): {macroA:.4f} <<<")
golds=[]; preds=[]
for (wd,P),r in zip(ensA,oof_rows):
    _,Y=gold_to_labels(r["final_text"]); golds.append(Y); preds.append(probs_to_multihot(wd,P,thresholds))
for i,m in enumerate(MARKS): print(f"   {m}: F1={per_class_f1(np.concatenate(golds),np.concatenate(preds))[i]:.3f}")

## 10. Train-combination whitelist + repair (subset backoff)

In [ ]:
# =========================================================================
#  Combinations = multi-hot mark set per gap. Whitelist those attested in train
#  gold; repair unseen predicted combos to their best seen SUBSET (removes hedged
#  marks, never invents). Honorific dashes are protected.
# =========================================================================
from collections import Counter
from itertools import combinations as _combs
def row_to_combo(r): return frozenset(np.where(r>0)[0].tolist())
def show_combo(c): return "∅" if not c else "".join(MARKS[i] for i in sorted(c))

train_combos=Counter()
for i in range(len(df)):
    _,Y=gold_to_labels(df.iloc[i]["final_text"])
    for r in Y: train_combos[row_to_combo(r)]+=1
SEEN={c for c,n in train_combos.items() if n>=MIN_COMBO_SUPPORT}
print(f"distinct train combos (support>={MIN_COMBO_SUPPORT}): {len(SEEN)}")
for c,n in sorted(train_combos.items(),key=lambda kv:-kv[1]):
    print(f"   {show_combo(c):<6}: {n}{'' if c in SEEN else '   (UNSEEN)'}")

def _subsets(s):
    idx=sorted(s)
    for r in range(len(idx),-1,-1):
        for c in _combs(idx,r): yield frozenset(c)

def repair_doc(words, P, thresholds):
    pred=(P>=thresholds[None,:]).astype(np.float32)
    pred=apply_honorific_rule(words,pred)
    if not DROP_UNSEEN_COMBOS: return pred
    di=M2I['-']
    for gi in range(pred.shape[0]):
        combo=row_to_combo(pred[gi])
        if combo in SEEN: continue
        forced=FORCE_HONORIFIC_RULE and pred[gi,di]>0 and (
            words[gi] in HONORIFICS or (gi+1<len(words) and words[gi+1] in HONORIFICS))
        best,bs=frozenset(),-1.0
        for sub in _subsets(combo):
            if sub not in SEEN: continue
            if forced and di not in sub: continue
            s=1.0
            for m in range(NUM_MARKS): s*= P[gi,m] if m in sub else (1.0-P[gi,m])
            if s>bs: bs,best=s,sub
        row=np.zeros(NUM_MARKS,np.float32)
        for m in best: row[m]=1.0
        pred[gi]=row
    return pred

# audit repair effect on OOF
_ch=0;_tot=0;rep=[]
for (wd,P),r in zip(ensA,oof_rows):
    b=probs_to_multihot(wd,P,thresholds); rr=repair_doc(wd,P,thresholds)
    _ch+=int((b!=rr).any(1).sum()); _tot+=len(wd); rep.append(reconstruct(wd,rr))
macro_rep=score(solA.copy(), pd.DataFrame({"id":range(len(df)),"final_text":rep}).copy(), "id")
print(f"\nOOF gaps altered by repair: {_ch}/{_tot} ({100*_ch/_tot:.3f}%)")
print(f">>> OOF macro-F1 after repair: {macro_rep:.4f} (was {macroA:.4f}) <<<")

## 11. Pseudo-label test (bag Stage-A folds → blend → repair)

In [ ]:
# =========================================================================
#  Bag Stage-A checkpoints on test, blend, confidence-gate (optional), repair.
# =========================================================================
assert os.path.exists(TEST_CSV), "test.csv not found — mount the competition dataset."
test=pd.read_csv(TEST_CSV)
id_col="id" if "id" in test.columns else test.columns[0]
test_rows=[{"text":t} for t in test["text"].tolist()]

print("bagging Stage-A folds on test:")
test_blend=bag_test(DIR_A, test_rows)

def confident(P):
    if CONF_GATE<=0: return P
    Q=P.copy(); Q[Q.max(1)<CONF_GATE]=0.0; return Q

pseudo_texts=[reconstruct(wd, repair_doc(wd, confident(P), thresholds)) for (wd,P) in test_blend]
pseudo_df=pd.DataFrame({id_col:test[id_col],"text":test["text"],"final_text":pseudo_texts})
pseudo_df.to_csv(os.path.join(PSEUDO_OUT,"test_pseudo.csv"), index=False)
# also drop a repaired Stage-A submission so you can submit BEFORE the retrain if you want
pseudo_df[[id_col,"final_text"]].to_csv("submission_stageA.csv", index=False)
print("wrote test_pseudo.csv + submission_stageA.csv", pseudo_df.shape)
print(pseudo_df.head(2).to_string())

## 12. STAGE B — build augmented train, fresh KFold(seed=2026), retrain

Real train + pseudo test are concatenated, then a **fresh** `KFold(seed=2026)` is drawn over the
**whole** combined set (so pseudo docs are spread and each is held out in its own fold). Pseudo ids
are prefixed to avoid collisions; an `is_pseudo` flag is kept for later ablation.

In [ ]:
# =========================================================================
#  Augmented dataframe + fresh KFold over the combined set (Stage B).
# =========================================================================
real=df[["id","text","final_text"]].copy(); real["is_pseudo"]=0
ps=pseudo_df.rename(columns={id_col:"id"})[["id","text","final_text"]].copy()
ps["id"]=PSEUDO_ID_PREFIX+ps["id"].astype(str); ps["is_pseudo"]=1

aug=pd.concat([real,ps], ignore_index=True)
fold_of=np.full(len(aug),-1,int)
kfB=KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for k,(_,va) in enumerate(kfB.split(np.arange(len(aug)))): fold_of[va]=k
aug["fold"]=fold_of
aug.to_csv(os.path.join(DIR_B,"train_plus_pseudo.csv"), index=False)
print("augmented:", aug.shape, "| real", len(real), "| pseudo", len(ps))
print("docs/fold:", aug["fold"].value_counts().sort_index().to_dict())
print("pseudo/fold:", aug[aug.is_pseudo==1]["fold"].value_counts().sort_index().to_dict())

# alignment sanity on the augmented set (pseudo rows must also be word-aligned)
_bad=sum(gold_to_labels(aug.iloc[i]["final_text"])[0]!=words_of_raw(aug.iloc[i]["text"]) for i in range(len(aug)))
print("augmented word-alignment mismatches (want 0):", _bad)

oofB = run_cv(aug, DIR_B, tag="B/augmented")

## 13. Stage-B OOF check + FINAL submission

In [ ]:
# =========================================================================
#  Re-tune thresholds on the Stage-B OOF (real+pseudo), report CV, then bag
#  Stage-B folds on test for the FINAL submission.
# =========================================================================
aug_rows=[aug.iloc[i] for i in range(len(aug))]
ensB=ensemble_oof(oofB, aug)
thresholds_B=tune_thresholds(ensB, aug_rows)
np.save(os.path.join(DIR_B,"thresholds.npy"), thresholds_B)
print("Stage-B thresholds:", {MARKS[i]:round(float(thresholds_B[i]),3) for i in range(NUM_MARKS)})

# CV reported on REAL docs only (pseudo labels aren't ground truth)
real_mask=(aug["is_pseudo"].values==0)
solB=pd.DataFrame([{"id":aug.iloc[i]["id"],"text":aug.iloc[i]["text"],"final_text":aug.iloc[i]["final_text"]}
                   for i in range(len(aug)) if real_mask[i]])
textsB=[reconstruct(wd, probs_to_multihot(wd,P,thresholds_B)) for i,(wd,P) in enumerate(ensB) if real_mask[i]]
subB=pd.DataFrame({"id":solB["id"].values,"final_text":textsB})
macroB=score(solB.copy(), subB.copy(), "id")
print(f"\n>>> Stage-B OOF host macro-F1 (real docs only): {macroB:.4f}  (Stage-A was {macroA:.4f}) <<<")

# FINAL: bag Stage-B folds on test
print("\nbagging Stage-B folds on test:")
final_blend=bag_test(DIR_B, test_rows)
final_texts=[reconstruct(wd, probs_to_multihot(wd,P,thresholds_B)) for (wd,P) in final_blend]
submission=pd.DataFrame({id_col:test[id_col],"final_text":final_texts})
submission.to_csv("submission.csv", index=False)
print("wrote submission.csv", submission.shape)
print(submission.head(2).to_string())
print("\nStage-A fallback submission also available: submission_stageA.csv")

## 14. Notes

- **Resuming:** `REUSE_SAVED=True` means a re-run skips any fold already checkpointed in `DIR_A` /
  `DIR_B`. If Kaggle times out, add the working dir as a dataset and re-run to continue.
- **CV honesty:** Stage-B macro-F1 is computed on **real docs only** — pseudo labels are model
  output, not ground truth, so scoring against them would be circular. Compare it to Stage-A to see
  whether pseudo-labeling actually helped before trusting the final submission.
- **If Stage-B doesn't beat Stage-A**, submit `submission_stageA.csv` (already written) — that's the
  repaired base ensemble with no pseudo augmentation.
- **Tuning the filter:** `MIN_COMBO_SUPPORT=2` treats train singletons (e.g. `؟!-`, `؟؛`) as unseen
  for stricter pseudo-labels; `CONF_GATE>0` blanks low-confidence gaps entirely.